In [ ]:
import os
import cv2
import numpy as np
from scipy import ndimage
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,accurc
from sklearn.utils import shuffle
from xgboost import XGBClassifier
import pickle
import pandas as pd
df = pd.DataFrame(columns=['number', 'feature', 'label'])

# Define the directories
dir_normal = 'C:\\Users\\yaswa\\learn\\Neural_networks\\research_ws\\Brain_ct_scan\\mal_dataset_thick\\mal_normal'
dir_thin = 'C:\\Users\\yaswa\\learn\\Neural_networks\\research_ws\\Brain_ct_scan\\mal_dataset_thick\\mal_acute'
dir_chronic = 'C:\\Users\\yaswa\\learn\\Neural_networks\\research_ws\\Brain_ct_scan\\mal_dataset_thick\\mal_chronic'

# Your WLD feature extraction function
def compute_wld(image):
    dx = ndimage.sobel(image.astype(float), 0)
    dy = ndimage.sobel(image.astype(float), 1)
    
    magnitude = np.sqrt(dx**2 + dy**2)
    orientation = np.arctan2(dy, dx) * (180.0 / np.pi) % 180.0
    
    avg_magnitude = ndimage.uniform_filter(magnitude, size=3)
    differential_excitation = np.abs(np.log1p(magnitude / (avg_magnitude + 1e-5)))
    
    num_excitation_bins = 8
    num_orientation_bins = 36
    differential_excitation_bins = np.floor(differential_excitation * num_excitation_bins)
    orientation_bins = np.floor(orientation / (180.0 / num_orientation_bins))
    
    wld_histogram, _ = np.histogram(2 * num_orientation_bins * differential_excitation_bins + orientation_bins, bins=num_excitation_bins * num_orientation_bins * 2, range=(0, num_excitation_bins * num_orientation_bins * 2))
    
    return wld_histogram

# Load and preprocess the images
patient,count=0,0
def load_images_from_directory(directory):
    
    for filename in natsorted(os.listdir(directory)):
        if filename.endswith(('.jpg', '.jpeg', '.png')):
            image_path = (os.path.join(directory, filename))
            image = augment_image(resize_image(cv2.imread(image_path)))
            label=get_class_label(os.listdir(directory))
            image=extract_patches(image,(128,128))
            num=str()
            if image is not None:
                df = df.append({'number': num, 'feature': feature, 'label': label}, ignore_index=True)
        


def get_class_label(patient_names):
    for names in patient_names:
        if 'acute' in names:
            labels.append(0)
        elif 'chronic' in names:
            labels.append(1)
        elif 'normal' in names:
            labels.append(2)
    return np.array(labels)

def resize_image(image, target_size=(512, 512)):
    resized_image = cv2.resize(image, target_size)
    return resized_image

def extract_patches(image, patch_size):
    patches = []
    height, width = image.shape[:2]
    for y in range(0, height - patch_size[0] + 1, patch_size[0]):
        for x in range(0, width - patch_size[1] + 1, patch_size[1]):
            patch = image[y:y+patch_size[0], x:x+patch_size[1]]
            patches.append(patch)
    return patches

# Load the data
images_normal, labels_normal = load_images(dir_normal, 0)
images_acute, labels_acute = load_images(dir_thin, 1)
images_chronic, labels_chronic = load_images(dir_chronic, 2)

# Combine the data
images = images_normal + images_acute + images_chronic
labels = labels_normal + labels_acute + labels_chronic

images, labels = shuffle(images, labels)

# Shuffle and split the data
X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42, shuffle=True)

# Train the model
model = XGBClassifier()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Print the classification report
print(classification_report(y_test, y_pred))

# Save the model
filename = 'mal_3_class_thick.pkl'
with open(filename, 'wb') as file:
    pickle.dump(model, file)

In [6]:
import os
import cv2
import numpy as np
from scipy import ndimage
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils import shuffle
from xgboost import XGBClassifier
import pickle
import pandas as pd

In [212]:
df=pd.read_excel('C:/Users/mahesh.inamdar/Desktop/codes/Feature list.xlsx',sheet_name='Scharr_Sig')

In [213]:
df.head(5)

,number,feature,label
0,AS_01_thin_1_1,[16384 0 0 0 0 0 0 ...,1
1,AS_01_thin_1_2,[9745 0 0 0 0 0 0 0 0 ...,1
2,AS_01_thin_1_3,[8160 0 0 0 0 0 0 0 0 ...,1
3,AS_01_thin_1_4,[15579 0 0 0 0 0 0 ...,1
4,AS_01_thin_1_5,[13247 0 0 0 0 0 0 ...,1


In [12]:
df1=df.drop(df.index[:3])
pd.to_numeric()

,Unnamed: 0,Unnamed: 1,Unnamed: 2
3,AS_01_thin_1_1,[16384 0 0 0 0 0 0 ...,1
4,AS_01_thin_1_2,[9424 5 5 11 4 8 4 3 3 ...,1
5,AS_01_thin_1_3,[7817 10 6 25 0 11 8 0 1 ...,1
6,AS_01_thin_1_4,[15508 2 1 2 1 1 0 ...,1
7,AS_01_thin_1_5,[13085 0 2 5 1 4 2 ...,1


In [214]:
X=df1.iloc[:,1].values
y=df1.iloc[:,2].values
y=y-1

In [200]:
def remove_special_characters(s):
    # Define a regular expression pattern to match non-digit characters
    pattern = r'[^0-9]'
    # Use the sub() function to replace non-digit characters with an empty string
    return re.sub(pattern, '', s)

In [216]:
XG=np.zeros(shape=(114312,700))

In [217]:
import re
X1=[i.split('\n') for i in X]
X2=[i for i in X1]
pe=[]
r,c=0,0
for a in range(114312):
    pe=[]
    for i in X2[a]:
        m=i.split(',')
        for j in m:
            n=j.split(' ')
            c=0
            for l in n:
                f=remove_special_characters(l)
                if not f:
                    continue
                else:
                    pe.append(f)
                    XG[r][c]=f
                    c+=1
    r+=1

In [218]:
from sklearn.metrics import classification_report,accuracy_score as acc
X_train, X_test, y_train, y_test = train_test_split(XG, y, test_size=0.3, random_state=42, shuffle=True)

# Train the model
model = XGBClassifier()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Print the classification report
print(classification_report(y_test.astype('int64'), y_pred, zero_division=1))

              precision    recall  f1-score   support

           0       1.00      0.00      0.00      3635
           1       0.42      0.01      0.02      5996
           2       0.72      1.00      0.84     24663

    accuracy                           0.72     34294
   macro avg       0.71      0.34      0.29     34294
weighted avg       0.70      0.72      0.60     34294



In [221]:
XG[13]

array([ 0., 26.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0